# Train the original polarity BERT model (Colab)

This notebook is the cleaned Colab version of `MachineLearning/NewsPolarityClassifier.ipynb`.
It follows the original ktrain/TensorFlow approach described in the PFE report.

**Steps**
1. Runtime → Change runtime type → GPU.
2. Run the install cell.
3. Upload `Data/GeneratedData/LabeledNews.csv` from this repository.
4. Run the training cells.
5. Download the generated `BertPolarityClassifier.zip`.
6. Follow `docs/TRAIN_POLARITY_MODEL.md` to install it in the repo.


In [ ]:
# ktrain pins are intentionally loose: use the TensorFlow version that Colab
# provides, then install ktrain. If resolution fails, see the troubleshooting
# section in docs/TRAIN_POLARITY_MODEL.md.
!pip install -q ktrain


In [ ]:
import ktrain
import pandas as pd
from ktrain import text
from sklearn.model_selection import train_test_split

print("ktrain", ktrain.__version__)


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose Data/GeneratedData/LabeledNews.csv


In [ ]:
data = pd.read_csv("LabeledNews.csv")
print(data.shape)
print(data.Sentiment.value_counts())
data.head()


In [ ]:
txt = data.News.astype(str).tolist()
sent = data.Sentiment.astype(str).tolist()

X_train, X_test, y_train, y_test = train_test_split(
    txt, sent, test_size=0.20, random_state=0, stratify=sent
)


In [ ]:
(x_train, y_train), (x_test, y_test), preproc = text.texts_from_array(
    x_train=X_train,
    y_train=y_train,
    x_test=X_test,
    y_test=y_test,
    class_names=["Negative", "Neutral", "Positive"],
    maxlen=350,
    preprocess_mode="bert",
)


In [ ]:
model = text.text_classifier(name="bert", train_data=(x_train, y_train), preproc=preproc)
learner = ktrain.get_learner(model, train_data=(x_train, y_train), batch_size=6)
learner.fit_onecycle(2e-5, 4)


In [ ]:
learner.validate(val_data=(x_test, y_test), class_names=["Negative", "Neutral", "Positive"])


In [ ]:
predictor = ktrain.get_predictor(learner.model, preproc)
predictor.predict("Apple beats earnings expectations as services revenue hits a record")


In [ ]:
predictor.save("/content/BertPolarityClassifier")

import shutil
shutil.make_archive("/content/BertPolarityClassifier", "zip", "/content/BertPolarityClassifier")
from google.colab import files
files.download("/content/BertPolarityClassifier.zip")


## Install the trained model in the repo

1. Unzip the download.
2. Copy the folder to `PythonObjects/legacy/PolarityClassifier/BertPolarityClassifier/`.
3. Use the legacy Python 3.10/ktrain environment and set `POLARITY_BACKEND=legacy` in `.env`.
4. Restart the service and check `/health`.

The default demo uses FinBERT and does not require this model.
